# Iberian Power Market: AI Day-Ahead Forecasting & Grid Transition
### Rystad Energy - Senior Data Engineer / Data Scientist Portfolio Project
---
This notebook analyzes the structural shift from fossil fuels to renewables in the Iberian grid and evaluates an XGBoost machine learning model trained to predict day-ahead market clearing prices.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# Set a professional dark theme suitable for trading/commodity dashboards
plt.style.use('dark_background')
sns.set_context("notebook", font_scale=1.1)

# Connect to the SQLite database and load the master view
conn = sqlite3.connect('energy_market.db')
df = pd.read_sql("SELECT * FROM energy_dashboard", conn)
df['time'] = pd.to_datetime(df['time'])
df.set_index('time', inplace=True)

# Create calculated columns for our analysis
df['Total Renewables'] = df['generation solar'] + df['generation wind onshore']
df['Total Fossil'] = df['generation fossil gas'] + df['generation fossil hard coal'] + df['generation fossil brown coal/lignite']

## 1. The Grid's Transition (Renewables vs Fossil Baseload)
Visualizing the structural shift and the intermittent nature of renewable generation compared to fossil fuel baseload.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Resample to daily averages for cleaner visualization
daily_gen = df[['Total Renewables', 'Total Fossil']].resample('D').mean()

ax.fill_between(daily_gen.index, daily_gen['Total Fossil'], color='#8c564b', alpha=0.7, label='Fossil Fuels (Gas & Coal)')
ax.fill_between(daily_gen.index, daily_gen['Total Renewables'], color='#2ca02c', alpha=0.8, label='Renewables (Solar & Wind)')

ax.set_title('The Iberian Grid Transition: Fossil vs Renewables (2015-2018)', fontsize=16, fontweight='bold')
ax.set_ylabel('Average Daily Generation (MW)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 2. ML Price Forecasting (XGBoost)
Tracking the day-ahead clearing price. We filter for the 2018 testing set where the XGBoost model made predictions.

In [ ]:
# Filter to only the testing period where predictions exist
test_df = df[df['predicted_price'].notnull()]

fig, ax = plt.subplots(figsize=(14, 6))

# Plotting the last 30 days of the dataset for a zoomed-in look at volatility
subset = test_df.tail(24 * 30)

ax.plot(subset.index, subset['price actual'], color='#1f77b4', label='Actual Spot Price', linewidth=2)
ax.plot(subset.index, subset['predicted_price'], color='#ff7f0e', label='XGBoost Predicted Price', linewidth=2, linestyle='--')

ax.set_title('Day-Ahead Market Forecasting (Last 30 Days)', fontsize=16, fontweight='bold')
ax.set_ylabel('Price (€/MWh)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.legend()
plt.tight_layout()
plt.show()

## 3. The Merit-Order Effect (Cannibalization)
Proving that high intermittent renewable generation (wind & solar) drives the commodity spot price down.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot with a regression line (trendline)
sns.regplot(
    data=df, 
    x='Total Renewables', 
    y='price actual', 
    scatter_kws={'alpha':0.1, 'color':'#17becf', 's': 10}, 
    line_kws={'color':'#d62728', 'linewidth':3}, 
    ax=ax
)

ax.set_title('Renewable Cannibalization Effect on Spot Prices', fontsize=16, fontweight='bold')
ax.set_xlabel('Total Renewable Generation (MW)')
ax.set_ylabel('Actual Spot Price (€/MWh)')
plt.tight_layout()
plt.show()